<a href="https://colab.research.google.com/github/FoozResq/Operational-Revenue-Intelligence-Platform/blob/main/Operational_Revenue_Intelligence_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retail Transaction RAG Pipeline

## Step 0 - Import Library

In [ ]:
import pandas as pd

##Step 1 — Data Ingestion


I'll use `pd.read_excel()` to load the data from your Excel file into a pandas DataFrame. You'll need to replace `'your_excel_file.xlsx'` with the actual path to your file.

In [ ]:
from google.colab import files
uploaded = files.upload()
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [ ]:
from datetime import datetime

excel_file_path = '/content/daily_sales.csv'
df_raw = pd.read_csv(excel_file_path, encoding="ISO-8859-1", dtype=str)
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(5)

In [ ]:
column_rename_map = {
    'ÇáãÓÊÎÏã': 'Saler',
    'ÅÌãÇáì': 'Total',
    'ÇáÝÑæÞ': 'Differences',
    'ÇáÅÔÊÑÇßÇÊ': 'Subscriptions',
    'ãÑÊÌÚÇÊ': 'Returns',
    'ãÈáÛ ÇáÝæÇÊíÑ': 'Invoice_Amount',
    'ãÈáÛ ÇáÕäÏæÞ': 'Cash_Amount',
    'ÇáÊÇÑíÎ': 'Date',
    'ã': 'Entry_ID' # Corrected column name
}

df_raw = df_raw.rename(columns=column_rename_map)

print("Columns after renaming:")
print(list(df_raw.columns))
display(df_raw.head())

## Step 2 — Data Quality Validation
## Step 3 — Governance Enforcement



In [ ]:
# ── Data profiling ───────────────────────────────────────────────────
print("=" * 55)
print("  RAW DATA QUALITY PROFILE")
print("=" * 55)

# Null counts per column
nulls = df_raw.isnull().sum()
print("\nNull counts per column:")
for col, n in nulls.items():
    pct = 100 * n / len(df_raw)
    bar = "█" * int(pct / 5)
    print(f"  {col:<15} {n:>7,}  ({pct:5.1f}%)  {bar}")

# Basic type conversion to inspect value ranges
df_raw["Total"]  = pd.to_numeric(df_raw["Total"],  errors="coerce")
df_raw["Invoice_Amount"] = pd.to_numeric(df_raw["Invoice_Amount"], errors="coerce")

print(f"\nTotal range     : {df_raw['Total'].min():,.2f}  →  {df_raw['Total'].max():,.2f}")
print(f"  Negative total          : {(df_raw['Total'] < 0).sum():,} rows")
print(f"  Zero total              : {(df_raw['Total'] == 0).sum():,} rows")

print(f"\nInvoice_Amount range : {df_raw['Invoice_Amount'].min():.2f}  →  {df_raw['Invoice_Amount'].max():.2f}")
print(f"  Negative invoice amount : {(df_raw['Invoice_Amount'] < 0).sum():,} rows")
print(f"  Zero invoice amount     : {(df_raw['Invoice_Amount'] == 0).sum():,} rows")

dupes = df_raw.duplicated().sum()
print(f"\nDuplicate rows  : {dupes:,}")

# Removed 'Country' and 'CustomerID' checks as these columns are not present

print("=" * 55)
print("\n💡 Key findings:")
print(f"   • {(df_raw['Total'] < 0).sum():,} rows have negative Total (e.g., returns or adjustments)")
print(f"   • {(df_raw['Invoice_Amount'] == 0).sum():,} rows have Invoice_Amount = 0 (e.g., no-charge items)")
print(f"   • {dupes:,} exact duplicate rows detected")

In [ ]:
#-------Data Contract (Pydantic v2)

import re
from pydantic import BaseModel, field_validator, ConfigDict
from datetime import datetime

class RetailTransactionContract(BaseModel):
    """
    Machine-enforceable schema for the Online Retail dataset.
    strict=True means no silent type coercion — '123' stays a string.
    """
    model_config = ConfigDict(strict=False)   # lenient for CSV strings

    saler: str
    total: float
    differences: float
    subscriptions: float
    returns: float
    invoice_amount: float
    cash_amount: float
    date: datetime
    entry_id: str

    @field_validator("entry_id")
    @classmethod
    def entry_id_required(cls, v: str) -> str:
        if not v or v.strip() in ("", "nan", "None"):
            raise ValueError("Entry_ID is required — cannot be null")
        return v.strip()

    @field_validator("invoice_amount")
    @classmethod
    def positive_invoice_amount(cls, v: float) -> float:
        if v <= 0:
            raise ValueError(f"invoice_amount must be > 0 (got {v} — likely a cancellation)")
        return v

    @field_validator("cash_amount")
    @classmethod
    def positive_cash_amount(cls, v: float) -> float:
        if v <= 0:
            raise ValueError(f"cash_amount must be > 0 (got {v} — likely a cancellation)")
        return v

    @field_validator("saler")
    @classmethod
    def valid_saler_format(cls, v: str) -> str:
        # Assuming 'saler' might be a name, the regex for 'InvoiceNo' doesn't fit.
        # Removing the regex if 'saler' is just a name, or adjust if there's a specific format.
        # For now, let's keep it simple if no specific format is defined for 'saler'.
        # If a specific format for 'saler' is required, please provide it.
        # The previous regex `r"^[A-Z]?\d{5,6}$"` is typically for invoice numbers, not names.
        # For demonstration, I will remove the regex if 'saler' is expected to be a name.
        # If 'saler' needs to conform to a specific ID format, please specify.
        # As no specific format is given for 'saler', I'm keeping the original regex for now, but it might not be appropriate.
        # if not re.match(r"^[A-Z]?\d{5,6}$", v.strip()):
        #     raise ValueError(f"saler format invalid: '{v}'")
        return v.strip()

print("✅ RetailTransactionContract defined.")

In [ ]:
# ── Apply contract validation ──────────────────────────────────────────────
!pip install loguru
from loguru import logger
logger.add("pipeline_execution.log", format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}")

print("Running contract validation on all rows (this may take ~30 seconds)...")

valid_rows, rejected_rows = [], []

for _, row in df_raw.iterrows():
    try:
        RetailTransactionContract(
            saler = str(row.get("Saler",  "") or ""),
            total = float(row.get("Total",  0)),
            differences = float(row.get("Differences", 0)),
            subscriptions = float(row.get("Subscriptions", 0)),
            returns = float(row.get("Returns", 0)),
            invoice_amount = float(row.get("Invoice_Amount", 0)),
            cash_amount = float(row.get("Cash_Amount", 0)),
            date = pd.to_datetime(row.get("Date", pd.NaT)), # Convert to datetime, handle NaT
            entry_id = str(row.get("Entry_ID", "") or ""),
        )
        valid_rows.append(row)
    except Exception as exc:
        row_copy = dict(row)
        row_copy["rejection_reason"] = str(exc)
        rejected_rows.append(row_copy)

df_valid    = pd.DataFrame(valid_rows)
df_rejected = pd.DataFrame(rejected_rows)

total = len(df_raw)
print(f"\n{'='*55}")
print(f"  CONTRACT VALIDATION RESULTS")
print(f"{'='*55}")
print(f"  Total rows    : {total:>10,}")
print(f"  ✅ Valid       : {len(df_valid):>10,}  ({100*len(df_valid)/total:.1f}%)")
print(f"  ❌ Rejected    : {len(df_rejected):>10,}  ({100*len(df_rejected)/total:.1f}%)")
print(f"{'='*55}")

if not df_rejected.empty:
    reasons = pd.DataFrame(df_rejected)["rejection_reason"]
    print("\nTop rejection reasons:")
    print(reasons.str.split(":").str[0].value_counts().head(5).to_string())

**DAMA 6-Dimension Quality Engine**

We run all six DAMA quality dimensions on the **contract-validated** subset.
These are not planted anomalies — they are real patterns in the data.

In [ ]:
#run all six DAMA quality dimensions on the **contract-validated** subset.
#These are not planted anomalies — they are real patterns in the data.

import json, uuid
from datetime import timedelta, datetime, UTC
import pandas as pd # Ensure pandas is imported if not already globally available

class DataQualityEngine:
    """
    Evaluates 6 DAMA dimensions on the Online Retail data:
      1. Completeness  — no nulls in critical fields
      2. Accuracy      — prices and quantities in valid ranges
      3. Consistency   — same InvoiceNo maps to same Country
      4. Timeliness    — all invoices within expected date range
      5. Uniqueness    — no full row duplicates
      6. Validity      — StockCode matches expected format
    """
    # Adjust expected date range to fit the observed data (2024-2026)
    EXPECTED_START = pd.Timestamp("2024-01-01")
    EXPECTED_END   = pd.Timestamp("2026-12-31") # Extend to cover known data

    def __init__(self, df: pd.DataFrame):
        self.df  = df.copy()
        self.log = {}

    def run(self) -> dict:
        logger.info("Starting DAMA 6-dimension scan on real retail data...")

        # 1. Completeness — Check 'Saler' for null or empty values
        missing_saler = (self.df["Saler"].isnull() | (self.df["Saler"].astype(str).str.strip() == "")).sum()
        self._record("1_completeness_saler",
                     missing_saler == 0,
                     f"{missing_saler:,} rows missing Saler")

        # 2. Accuracy — Check 'Total' for unusually high values
        # Using 10,000 as a placeholder high value, adjust if needed
        high_total = (self.df["Total"] > 10_000).sum()
        self._record("2_accuracy_total",
                     high_total == 0,
                     f"{high_total:,} rows with Total > 10,000 (potential data entry error)")

        # 3. Consistency — (Removed due to missing 'InvoiceNo' and 'Country' columns)
        # This dimension is skipped as there's no direct equivalent in the current dataset.
        # If a consistency rule for existing columns is desired, it needs to be defined.
        self._record("3_consistency", True, "Consistency check for InvoiceNo-Country skipped (columns not present).")

        # 4. Timeliness — 'Date' within expected range
        if "Date" in self.df.columns:
            try:
                dates = pd.to_datetime(self.df["Date"], errors="coerce")
                out_of_range = ((dates < self.EXPECTED_START) | (dates > self.EXPECTED_END)).sum()
                self._record("4_timeliness_date",
                             out_of_range == 0,
                             f"{out_of_range:,} invoices outside {self.EXPECTED_START.year}–{self.EXPECTED_END.year} range")
            except Exception:
                self._record("4_timeliness_date", False, "Could not parse Date column for timeliness check")
        else:
            self._record("4_timeliness_date", False, "'Date' column not found for timeliness check")

        # 5. Uniqueness — no fully duplicated rows
        dupes = self.df.duplicated().sum()
        self._record("5_uniqueness_rows",
                     dupes == 0,
                     f"{dupes:,} exact duplicate rows")

        # 6. Validity — 'Entry_ID' should be 5 digits
        # Assuming Entry_ID should be a 5-digit number based on sample data like '43931', '54041'
        invalid_entry_id = (~self.df["Entry_ID"].astype(str).str.match(
            r"^\d{5}$", na=False
        )).sum()
        self._record("6_validity_entry_id",
                     invalid_entry_id == 0,
                     f"{invalid_entry_id:,} Entry_IDs don't match 5-digit numeric format")

        passed = all(v["passed"] for v in self.log.values() if "skipped" not in v["detail"])
        return {"passed": passed, "dimensions": self.log,
                "timestamp": datetime.now(UTC).isoformat(), "row_count": len(self.df)}

    def _record(self, name, passed, detail):
        self.log[name] = {"passed": passed,
                          "status": "PASSED" if passed else "FAILED",
                          "detail": detail}
        if passed:
            logger.success(f"  [PASSED] {name}: {detail}")
        else:
            logger.warning(f"  [FAILED] {name}: {detail}")

# Run the engine
dq = DataQualityEngine(df_valid)
report = dq.run()

print("\n" + "="*55)
print("  DAMA DATA QUALITY REPORT")
print("="*55)
for dim, result in report["dimensions"].items():
    icon = "✅" if result["passed"] else "❌"
    print(f"  {icon}  {dim}")
    print(f"       {result['detail']}")
print("="*55)
print(f"\nOverall verdict: {'PASSED ✅' if report['passed'] else 'FAILED ❌'}")
print(f"Rows evaluated : {report['row_count']:,}")

  **Route: Production Warehouse or Quarantine**

Based on the quality verdict, we either promote the data or quarantine it.
Regardless of the verdict, we always write the rejected rows to quarantine
so producers can fix them.

In [ ]:
import os
from datetime import datetime, UTC

os.makedirs("production_warehouse", exist_ok=True)
os.makedirs("quarantine_zone",      exist_ok=True)

# Always quarantine contract-rejected rows
if not df_rejected.empty:
    q_path = f"quarantine_zone/contract_violations_{int(datetime.now(UTC).timestamp())}.csv"
    pd.DataFrame(df_rejected).to_csv(q_path, index=False)
    logger.error(f"{len(df_rejected):,} contract violations → {q_path}")

# Route based on DAMA verdict
if report["passed"]:
    out = "production_warehouse/online_retail_clean.parquet"
    df_valid.to_parquet(out, index=False)
    logger.success(f"Quality gate PASSED — {len(df_valid):,} rows → {out}")
    print(f"\n✅ Production write: {out}")
    print(f"   {len(df_valid):,} clean rows ready for downstream analytics")
else:
    q_path = f"quarantine_zone/dq_failure_{int(datetime.now(UTC).timestamp())}.csv"
    df_valid.to_csv(q_path, index=False)
    logger.error(f"Quality gate FAILED — {len(df_valid):,} rows → {q_path}")
    print(f"\n❌ Quality gate failed — data quarantined to {q_path}")
    print("   Fix the flagged dimensions and re-run the pipeline.")

**OpenLineage Audit Trail**

Every pipeline run emits structured lineage events.
In production these would be sent to a **Marquez** server via HTTP POST.
Here we write them locally for inspection.

In [ ]:
import json, uuid, os
from datetime import datetime, UTC

class LineageEmitter:
    def __init__(self, job_name):
        self.job_name = job_name
        self.run_id   = str(uuid.uuid4())
        self.events   = []

    def emit(self, event_type, input_ds, output_ds, facets=None):
        self.events.append({
            "eventType": event_type,
            "eventTime": datetime.now(UTC).isoformat(),
            "run":    {"runId": self.run_id},
            "job":    {"namespace": "day4_lab", "name": self.job_name},
            "inputs":  [{"namespace": "kaggle", "name": input_ds}],
            "outputs": [{"namespace": "lab",    "name": output_ds,
                         "facets": facets or {}}],
        })
        print(f"[LINEAGE] {event_type:10s} | {input_ds} → {output_ds}")

    def write(self, path="lineage_events/run.json"):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w") as f:
            json.dump(self.events, f, indent=2)
        print(f"\n📋 {len(self.events)} lineage events → {path}")


lineage = LineageEmitter("retail_quality_pipeline")

lineage.emit("START",    "online_retail_raw.csv", "landing_zone",
             {"row_count": len(df_raw)})

lineage.emit("COMPLETE" if df_rejected.empty else "FAIL",
             "landing_zone", "quarantine_zone/contract_violations",
             {"rejected_rows": len(df_rejected),
              "rejection_rate_pct": round(100 * len(df_rejected) / len(df_raw), 2)})

if report["passed"]:
    lineage.emit("COMPLETE", "landing_zone", "production_warehouse/online_retail_clean",
                 {"row_count": len(df_valid),
                  "dama_dimensions_passed": sum(1 for v in report["dimensions"].values() if v["passed"]),
                  "dama_dimensions_total":  len(report["dimensions"])})
else:
    failed_dims = [k for k, v in report["dimensions"].items() if not v["passed"]]
    lineage.emit("FAIL", "landing_zone", "quarantine_zone/dq_failure",
                 {"row_count": len(df_valid), "failed_dimensions": failed_dims})

lineage.write()

**Explore the Clean Data**

Now let's actually look at what passed the quality gates and draw some insights.

In [ ]:
if report["passed"] and os.path.exists("production_warehouse/online_retail_clean.parquet"):
    df_clean = pd.read_parquet("production_warehouse/online_retail_clean.parquet")
else:
    df_clean = df_valid.copy()

# Calculate Revenue using 'Invoice_Amount' as a proxy, since 'Quantity' and 'UnitPrice' are not available
df_clean["Revenue"] = df_clean["Invoice_Amount"].astype(float) # Using Invoice_Amount as Revenue

print("📊 Top 10 Salers by revenue (clean data only):") # Changed "countries" to "Salers"
top_salers = ( # Changed variable name to reflect 'Salers'
    df_clean.groupby("Saler")["Revenue"] # Group by "Saler" instead of "Country"
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print(top_salers.apply(lambda x: f"  £{x:>12,.2f}").to_string())

# The following lines rely on columns that do not exist in the current dataset (StockCode, CustomerID, InvoiceNo).
# They have been removed as they cannot be computed with the available data.
# print(f"\n📦 Unique products  : {df_clean['StockCode'].nunique():,}")
# print(f"👥 Unique customers : {df_clean['CustomerID'].nunique():,}")
# print(f"🧾 Unique invoices  : {df_clean['InvoiceNo'].nunique():,}")
print(f"\n💷 Total revenue    : £{df_clean['Revenue'].sum():,.2f}")

print("\n✅ Pipeline complete.")
print(f"   Started with {len(df_raw):,} raw rows.")
print(f"   {len(df_rejected):,} rejected by contract ({100*len(df_rejected)/len(df_raw):.1f}%).")
print(f"   {len(df_clean):,} rows in production warehouse ready for analytics.")

In [ ]:
print("📊 Top 10 Days by Revenue (clean data only):")
top_days_revenue = (
    df_clean.groupby("Date")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print(top_days_revenue.apply(lambda x: f"  £{x:>12,.2f}").to_string())

## Step 4 — Data Storage


In [ ]:
!pip install pyspark==4.0.1 delta-spark==4.3.0

In [ ]:
import asyncio
import os
import random
import shutil
from datetime import datetime, timedelta, timezone
from collections import defaultdict
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
from delta import configure_spark_with_delta_pip

**1. BACKBONE — Mock Kafka Broker (partitions + offsets)**

In [ ]:
# =====================================================================

class MockKafkaBroker:
    """
    Simulates Apache Kafka's core model.

    In real Kafka:
      - A topic is split into N partitions (ordered, immutable logs).
      - Each message gets an offset — its address within the partition.
      - A consumer group shares partitions; each partition goes to
        exactly one member, so adding consumers beyond the partition
        count yields no extra throughput.
      - Committed offsets let consumers resume after a crash.
    """

    def __init__(self):
        self.topics  = defaultdict(asyncio.Queue)
        self.offsets = defaultdict(int)

    async def publish(self, topic: str, message: dict):
        offset = self.offsets[topic]
        self.offsets[topic] += 1
        await self.topics[topic].put({
            "offset":     offset,
            "event_time": datetime.now(timezone.utc).isoformat(),
            "topic":      topic,
            "payload":    message,
        })

    async def consume(self, topic: str) -> dict:
        return await self.topics[topic].get()


def init_delta_spark() -> SparkSession:
    builder = (
        SparkSession.builder
        .appName("Day2_RealTimeStreamingPipeline")
        .master("local[*]")
        .config("spark.sql.extensions",
                "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.warehouse.dir", "./data/warehouse")
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    return spark

**2. INGESTION LAYER — Three concurrent IoT producers**

In [ ]:
# =====================================================================

async def machine_producer(broker: MockKafkaBroker, machine_id: str):
    """
    Simulates a factory sensor streaming temperature readings every 300ms.
    12% chance of a spike anomaly to trigger downstream alerts.
    """
    print(f"  🏭 [PRODUCER] {machine_id} stream online.")
    while True:
        is_spike = random.random() < 0.12
        temp = random.uniform(92.0, 118.0) if is_spike else random.uniform(58.0, 78.0)
        await broker.publish("factory_sensors_raw", {
            "machine_id": machine_id,
            "temperature": round(temp, 2),
            "event_ts":   datetime.now(timezone.utc).isoformat(),
        })
        await asyncio.sleep(0.3)

**3. COMPUTE LAYER — Stream processor with window types + watermark**

In [ ]:
# =====================================================================

class StreamProcessor:
    """
    Stateful stream processing engine — mirrors Flink / Spark Structured Streaming.

    Three window types demonstrated:
      TUMBLING   — fixed, non-overlapping 1-second buckets.
                   Every event belongs to exactly one bucket.
                   Use case: per-second event count, hourly totals.

      SLIDING    — overlapping rolling window over the last 3 seconds.
                   The same event can appear in multiple windows.
                   Use case: rolling average temperature or CPU.

      WATERMARK  — events older than WATERMARK_S seconds are dropped.
                   This bounds state memory and lets the engine close
                   windows cleanly. Larger watermark = more completeness
                   but higher latency.
    """

    WATERMARK_S      = 2
    SLIDING_WINDOW_S = 3

    def __init__(self, broker: MockKafkaBroker, threshold: float):
        self.broker    = broker
        self.threshold = threshold
        self.sliding   = defaultdict(list)
        self.tumbling  = defaultdict(list)
        self.watermark = datetime.now(timezone.utc)

    async def run(self):
        print(f"  🧠 [PROCESSOR] Window engine live. Threshold: {self.threshold}°C")
        while True:
            event   = await self.broker.consume("factory_sensors_raw")
            payload = event["payload"]
            m_id    = payload["machine_id"]
            val     = payload["temperature"]
            now     = datetime.now(timezone.utc)

            # Watermark — drop late events
            if now < self.watermark - timedelta(seconds=self.WATERMARK_S):
                print(f"  ⚠️  [WATERMARK] Late event from {m_id} dropped.")
                continue
            self.watermark = max(self.watermark, now)

            # Sliding window — rolling average
            self.sliding[m_id].append((now, val))
            cutoff = now - timedelta(seconds=self.SLIDING_WINDOW_S)
            self.sliding[m_id] = [
                (t, v) for t, v in self.sliding[m_id] if t > cutoff
            ]
            rolling_avg = (
                sum(v for _, v in self.sliding[m_id]) / len(self.sliding[m_id])
            )

            # Tumbling window — count events in this 1-second bucket
            bucket = now.strftime("%H:%M:%S")
            self.tumbling[bucket].append(val)
            bucket_count = len(self.tumbling[bucket])

            # Threshold check
            if val > self.threshold:
                await self.broker.publish("anomalies_sink", {
                    "machine_id":   m_id,
                    "value":        val,
                    "rolling_avg":  round(rolling_avg, 2),
                    "bucket":       bucket,
                    "bucket_count": bucket_count,
                    "status":       "CRITICAL",
                })

**4. STORAGE SINK — ACID append to Delta Lake**

In [ ]:
# =====================================================================

async def delta_sink(
    broker: MockKafkaBroker,
    spark: SparkSession,
    delta_path: str,
):
    """
    Reads anomaly alerts and commits them to Delta Lake.
    Delta's transaction log guarantees ACID — no partial writes,
    no corruption even if two writers race on the same files.
    """
    schema = StructType([
        StructField("event_time",   StringType(),  True),
        StructField("machine_id",   StringType(),  True),
        StructField("value",        DoubleType(),  True),
        StructField("rolling_avg",  DoubleType(),  True),
        StructField("bucket",       StringType(),  True),
        StructField("bucket_count", IntegerType(), True),
        StructField("status",       StringType(),  True),
    ])
    print("  💾 [SINK] Delta Lake writer active.")
    while True:
        alert = await broker.consume("anomalies_sink")
        p = alert["payload"]
        row = [(
            alert["event_time"],
            p["machine_id"],
            p["value"],
            p["rolling_avg"],
            p["bucket"],
            p["bucket_count"],
            p["status"],
        )]
        spark.createDataFrame(row, schema).write \
             .format("delta").mode("append").save(delta_path)
        print(f"  🚨 [DELTA WRITE] {p['machine_id']} — "
              f"{p['value']}°C | rolling avg {p['rolling_avg']}°C | "
              f"bucket {p['bucket']} ({p['bucket_count']} events)")

In [ ]:
# Initialize SparkSession for Delta Lake operations
spark = init_delta_spark()

# Define a schema for df_clean for PySpark, inferring types where possible
# For simplicity, let's map pandas types to Spark types directly.
# Ensure 'Date' is handled as TimestampType if it's a datetime object in df_clean.
# If 'Date' is still a string in df_clean, it should be StringType or parsed first.
# Based on the kernel state, df_clean['Date'] is likely a string or object that can be converted to TimestampType.
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

clean_schema = StructType([
    StructField("Saler", StringType(), True),
    StructField("Total", DoubleType(), True),
    StructField("Differences", DoubleType(), True),
    StructField("Subscriptions", DoubleType(), True),
    StructField("Returns", DoubleType(), True),
    StructField("Invoice_Amount", DoubleType(), True),
    StructField("Cash_Amount", DoubleType(), True),
    StructField("Date", TimestampType(), True), # Assuming Date is convertible to TimestampType
    StructField("Entry_ID", StringType(), True),
    StructField("Revenue", DoubleType(), True)
])

# Convert df_clean (pandas DataFrame) to Spark DataFrame
# Ensure 'Date' column in pandas DataFrame is of datetime type before converting to Spark
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

# Convert columns to numeric types in pandas DataFrame before creating Spark DataFrame
for col in ["Differences", "Subscriptions", "Returns", "Cash_Amount", "Total", "Invoice_Amount", "Revenue"]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

spark_df_clean = spark.createDataFrame(df_clean, schema=clean_schema)

# Define the Delta Lake path for the clean data
clean_delta_path = "./data/warehouse/clean_retail_transactions"

# Write the Spark DataFrame to Delta Lake
spark_df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(clean_delta_path)

print(f"✅ Successfully wrote df_clean to Delta Lake at: {clean_delta_path}")
print(f"Number of rows written: {spark_df_clean.count()}")

In [ ]:
# To verify, you can read the data back from Delta Lake
print("\nVerifying data written to Delta Lake...")
df_read_delta = spark.read.format("delta").load(clean_delta_path)
df_read_delta.show()

## Step 5 — Embedding Generation


In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained sentence transformer model
# 'all-MiniLM-L6-v2' is a good balance of speed and performance
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Embedding model loaded.")

Now, let's define some example texts related to our retail transactions and generate their embeddings.

In [ ]:
# Example retail-related phrases for embedding
example_texts = [
    "Evening Cashier processed a large sale on October 9, 2025.",
    "A customer returned an item, resulting in a negative total.",
    "Morning Cashier handled several subscriptions today.",
    "Invoice for product 54055 shows a total amount of 924.89.",
    "Anomaly detected: very high temperature reading from machine_id A."
]

# Generate embeddings for the example texts
embeddings = embedding_model.encode(example_texts)

print(f"Generated embeddings for {len(example_texts)} texts.")
print(f"Each embedding has a dimension of: {embeddings.shape[1]}")

# Display the first embedding to show its structure
print("\nFirst embedding (truncated):")
print(embeddings[0][:10]) # Displaying first 10 dimensions of the first embedding

In [ ]:
# Prepare text for embedding from df_clean
# Concatenate relevant columns into a single string for each row
# Convert numerical columns to string to include them in the text representation

df_clean_for_embedding = df_clean.copy()

df_clean_for_embedding['text_for_embedding'] = df_clean_for_embedding.apply(
    lambda row: (
        f"Saler: {row['Saler']}. "
        f"Total: {row['Total']:.2f}. "
        f"Differences: {row['Differences']:.2f}. "
        f"Subscriptions: {row['Subscriptions']:.2f}. "
        f"Returns: {row['Returns']:.2f}. "
        f"Invoice Amount: {row['Invoice_Amount']:.2f}. "
        f"Cash Amount: {row['Cash_Amount']:.2f}. "
        f"Date: {row['Date'].strftime('%Y-%m-%d')}. " # Format date for consistency
        f"Entry ID: {row['Entry_ID']}. "
        f"Revenue: {row['Revenue']:.2f}."
    ), axis=1
)

# Generate embeddings for the 'text_for_embedding' column
print(f"Generating embeddings for {len(df_clean_for_embedding)} rows...")
embeddings_for_df_clean = embedding_model.encode(df_clean_for_embedding['text_for_embedding'].tolist(), show_progress_bar=True)

# Add embeddings and text_for_embedding as new columns to df_clean
df_clean['embeddings'] = embeddings_for_df_clean.tolist()
df_clean['text_for_embedding'] = df_clean_for_embedding['text_for_embedding'] # Add text_for_embedding to df_clean

print(f"\nGenerated embeddings for {len(df_clean)} rows.")
print(f"Each embedding has a dimension of: {embeddings_for_df_clean.shape[1]}")

print("\nFirst 5 rows of df_clean with new 'embeddings' and 'text_for_embedding' columns:")
display(df_clean.head())

In [ ]:
import chromadb

# Initialize ChromaDB client
# For simplicity, we'll use an an ephemeral client. For persistence, you'd use chromadb.PersistentClient()
client = chromadb.Client()

# Define the collection name
collection_name = "retail_transactions_embeddings"

# If the collection already exists, delete it to ensure a clean start
# This is useful for re-running the notebook
try:
    client.delete_collection(name=collection_name)
    print(f"Existing collection '{collection_name}' deleted.")
except:
    print(f"Collection '{collection_name}' does not exist, creating a new one.")

# Create a new collection
# We specify the embedding function as we're bringing our own embeddings
collection = client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"} # Cosine distance is common for sentence embeddings
)

# Prepare data for insertion
# IDs should be unique strings
ids = [str(e_id) for e_id in df_clean['Entry_ID'].tolist()]

# Embeddings are already generated and in df_clean['embeddings']
embeddings_to_store = df_clean['embeddings'].tolist()

# Metadatas to store alongside embeddings (optional but highly recommended)
# Include the raw text that generated the embedding for context
metadatas = [
    {"saler": row['Saler'], "total": row['Total'], "date": row['Date'].strftime('%Y-%m-%d'), "entry_id": str(row['Entry_ID']), "text": row['text_for_embedding']}
    for _, row in df_clean_for_embedding.iterrows()
]

# Documents to store alongside embeddings (the actual text content)
documents_to_store = df_clean_for_embedding['text_for_embedding'].tolist()

# Add documents to the collection
print(f"Adding {len(embeddings_to_store)} embeddings to ChromaDB collection '{collection_name}'...")
collection.add(
    embeddings=embeddings_to_store,
    metadatas=metadatas,
    documents=documents_to_store, # Explicitly add documents
    ids=ids
)

print(f"✅ Successfully added {collection.count()} items to the collection '{collection_name}'.")

## Step 6 — Vector Storage


**Vector Databases and Advanced RAG Engineering**

Full RAG pipeline: document chunking → ChromaDB (HNSW) → BM25 → RRF hybrid search → cross-encoder reranking → RAGAS-style evaluation.

In [ ]:
!pip install chromadb sentence-transformers rank-bm25 numpy

In [ ]:
import chromadb

# Initialize ChromaDB client
# For simplicity, we'll use an ephemeral client. For persistence, you'd use chromadb.PersistentClient()
client = chromadb.Client()

# Define the collection name
collection_name = "retail_transactions_embeddings"

# If the collection already exists, delete it to ensure a clean start
# This is useful for re-running the notebook
try:
    client.delete_collection(name=collection_name)
    print(f"Existing collection '{collection_name}' deleted.")
except:
    print(f"Collection '{collection_name}' does not exist, creating a new one.")

# Create a new collection
# We specify the embedding function as we're bringing our own embeddings
collection = client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"} # Cosine distance is common for sentence embeddings
)

# Prepare data for insertion
# IDs should be unique strings
ids = [str(e_id) for e_id in df_clean['Entry_ID'].tolist()]

# Embeddings are already generated and in df_clean['embeddings']
embeddings_to_store = df_clean['embeddings'].tolist()

# Metadatas to store alongside embeddings (optional but highly recommended)
# Include the raw text that generated the embedding for context
metadatas = [
    {"saler": row['Saler'], "total": row['Total'], "date": row['Date'].strftime('%Y-%m-%d'), "text": row['text_for_embedding']}
    for _, row in df_clean_for_embedding.iterrows()
]

# Add documents to the collection
print(f"Adding {len(embeddings_to_store)} embeddings to ChromaDB collection '{collection_name}'...")
collection.add(
    embeddings=embeddings_to_store,
    metadatas=metadatas,
    ids=ids
)

print(f"✅ Successfully added {collection.count()} items to the collection '{collection_name}'.")

In [ ]:
import chromadb

# Initialize ChromaDB client
# For simplicity, we'll use an ephemeral client. For persistence, you'd use chromadb.PersistentClient()
client = chromadb.Client()

# Define the collection name
collection_name = "retail_transactions_embeddings"

# If the collection already exists, delete it to ensure a clean start
# This is useful for re-running the notebook
try:
    client.delete_collection(name=collection_name)
    print(f"Existing collection '{collection_name}' deleted.")
except:
    print(f"Collection '{collection_name}' does not exist, creating a new one.")

# Create a new collection
# We specify the embedding function as we're bringing our own embeddings
collection = client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"} # Cosine distance is common for sentence embeddings
)

# Prepare data for insertion
# IDs should be unique strings
ids = [str(e_id) for e_id in df_clean['Entry_ID'].tolist()]

# Embeddings are already generated and in df_clean['embeddings']
embeddings_to_store = df_clean['embeddings'].tolist()

# Metadatas to store alongside embeddings (optional but highly recommended)
# Include the raw text that generated the embedding for context
metadatas = [
    {"saler": row['Saler'], "total": row['Total'], "date": row['Date'].strftime('%Y-%m-%d'), "text": row['text_for_embedding']}
    for _, row in df_clean_for_embedding.iterrows()
]

# Add documents to the collection
print(f"Adding {len(embeddings_to_store)} embeddings to ChromaDB collection '{collection_name}'...")
collection.add(
    embeddings=embeddings_to_store,
    metadatas=metadatas,
    ids=ids
)

print(f"✅ Successfully added {collection.count()} items to the collection '{collection_name}'.")

## Step 7 — RAG Retrieval


In [ ]:
# Removed: !pip install groq

**Groq API Key Setup**
To use the Groq API, you'll need an API key. Obtain one from the Groq Console.

Once you have your key, add it to Colab's secret manager. Click the "🔑" icon in the left panel, then select "Add a new secret." Name the secret GROQ_API_KEY and paste your key there. Ensure "Notebook access" is enabled for this secret.

In [ ]:
import os
import json
import uuid
import random
from datetime import datetime, timedelta, UTC # Added UTC
from collections import defaultdict

import pandas as pd
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import SentenceTransformer
from loguru import logger

logger.add("capstone_run.log", rotation="10 MB")

BRONZE_PATH = "./lakehouse/bronze"
SILVER_PATH = "./lakehouse/silver"
GOLD_PATH   = "./lakehouse/gold"

In [ ]:
# Removed: Groq client initialization
# from groq import Groq
# from google.colab import userdata

# try:
#     GROQ_API = userdata.get('GROQ_API_KEY')
#     groq_client = Groq(api_key=GROQ_API)
#     logger.info('Groq client initialized successfully.')
# except Exception as e:
#     groq_client = None
#     logger.error(f'Failed to initialize Groq client: {e}. Make sure GROQ_API_KEY is set in Colab secrets.')

### Retrieval Augmented Generation (RAG) Setup

Now we'll define a function to perform RAG. This involves:
1. Embedding the user's query.
2. Using the query embedding to find relevant documents in our ChromaDB collection.
3. Passing these retrieved documents (context) along with the original query to the Groq LLM to generate a response.

In [ ]:
import re

def retrieve_context(query: str, top_k: int = 3):
    """Embeds a query and retrieves the top_k most relevant documents from ChromaDB, with optional Entry ID filtering."""
    query_embedding = embedding_model.encode([query]).tolist()

    # Try to extract an Entry ID from the query using regex
    entry_id_match = re.search(r'entry ID (\d+)', query, re.IGNORECASE)
    where_clause = {}
    if entry_id_match:
        extracted_entry_id = entry_id_match.group(1)
        where_clause = {"entry_id": extracted_entry_id}
        print(f"🔍 Detected Entry ID '{extracted_entry_id}' in query. Applying metadata filter.")

    # Perform the query with or without the where clause
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=['metadatas', 'documents'],
        where=where_clause if where_clause else None # Apply filter if Entry ID was extracted
    )

    # Extract metadatas and documents for easier use
    retrieved_contexts = []
    if results and results['metadatas'] and results['documents']:
        for i in range(len(results['metadatas'][0])):
            context_item = {
                'metadata': results['metadatas'][0][i],
                'document': results['documents'][0][i]
            }
            retrieved_contexts.append(context_item)
    return retrieved_contexts

print("✅ `retrieve_context` function defined and enhanced for Entry ID filtering.")

Now, let's define a function to generate an AI response using the Groq LLM, incorporating the retrieved context.

In [ ]:
# Removed: generate_ai_response function as it depended on Groq.
# You can implement your own LLM integration here to process the retrieved_contexts.
# def generate_ai_response(query: str, retrieved_contexts: list):
#     """Generates an AI response using Groq, augmented with retrieved contexts."""
#     if groq_client is None:
#         logger.error("Groq client is not initialized. Cannot generate AI response.")
#         return "Error: Groq client not initialized. Please ensure GROQ_API_KEY is set in Colab secrets."

#     context_str = "\n\n".join([item['document'] for item in retrieved_contexts])

#     system_prompt = (
#         "You are an AI assistant specialized in retail transaction analysis. "
#         "Answer the user's query based *only* on the provided retail transaction data. "
#         "If the answer is not in the context, state that you don't have enough information. "
#         "Be concise and directly answer the question."
#     )

#     user_message = f"Based on the following retail transaction data:\n\n{context_str}\n\nAnswer this question: {query}"

#     try:
#         chat_completion = groq_client.chat.completions.create(
#             messages=[
#                 {
#                     "role": "system",
#                     "content": system_prompt,
#                 },
#                 {
#                     "role": "user",
#                     "content": user_message,
#                 }
#             ],
#             model="llama-3.1-8b-instant",
#                     temperature=0,
#                     max_tokens=150,
#         )
#         return chat_completion.choices[0].message.content
#     except Exception as e:
#         logger.error(f"Failed to generate AI response from Groq: {e}")
#         return f"Error generating response: {e}"

print("✅ `generate_ai_response` function removed. Please implement your own LLM integration.")

 Demo: Ask a question to the RAG system

In [ ]:
query = "What was the total amount for entry ID 42670 and who was the saler?"
print(f"User Query: {query}")

# Step 1: Retrieve relevant context
retrieved_contexts = retrieve_context(query, top_k=2)
print("\n--- Retrieved Contexts ---")
for i, item in enumerate(retrieved_contexts):
    print(f"Context {i+1}: {item['document']}")
    # print(f"  Metadata: {item['metadata']}") # Uncomment to see metadata

# Removed: Step 2: Generate AI response using the retrieved context
# To use an LLM, you would call your custom `generate_ai_response` function here
# For now, we will just display the retrieved contexts.
print("\n--- Retrieved Contexts for LLM processing ---")
for i, item in enumerate(retrieved_contexts):
    print(f"Context {i+1}:\n{item['document']}\n")

print("Please integrate your preferred LLM to process these retrieved contexts and generate a response.")

In [ ]:
query = "What was the date for entry ID 42670?"
print(f"User Query: {query}")

# Step 1: Retrieve relevant context
retrieved_contexts = retrieve_context(query, top_k=2)
print("\n--- Retrieved Contexts ---")
for i, item in enumerate(retrieved_contexts):
    print(f"Context {i+1}: {item['document']}")
    # print(f"  Metadata: {item['metadata']}") # Uncomment to see metadata

# Removed: Step 2: Generate AI response using the retrieved context
# To use an LLM, you would call your custom `generate_ai_response` function here
# For now, we will just display the retrieved contexts.
print("\n--- Retrieved Contexts for LLM processing ---")
for i, item in enumerate(retrieved_contexts):
    print(f"Context {i+1}:\n{item['document']}\n")

print("Please integrate your preferred LLM to process these retrieved contexts and generate a response.")

In [ ]:
entry_id_to_verify = '42670'
retrieved_from_chroma_by_id = collection.get(ids=[entry_id_to_verify], include=['metadatas', 'documents'])

if retrieved_from_chroma_by_id and retrieved_from_chroma_by_id['ids']:
    print(f"✅ Entry ID {entry_id_to_verify} found in ChromaDB:")
    print(f"  Metadata: {retrieved_from_chroma_by_id['metadatas'][0]}")
    print(f"  Document: {retrieved_from_chroma_by_id['documents'][0]}")
else:
    print(f"❌ Entry ID {entry_id_to_verify} NOT found in ChromaDB.")

# Also, let's try increasing top_k in the semantic search to see if it eventually finds it
print(f"\nAttempting RAG retrieval with increased top_k to 5 for query: {query}")
retrieved_contexts_higher_k = retrieve_context(query, top_k=5)
print("--- Retrieved Contexts (top_k=5) ---")
for i, item in enumerate(retrieved_contexts_higher_k):
    print(f"Context {i+1}: {item['document']}")

In [ ]:
entry_id_to_check = '42670'
entry_id_data = df_clean[df_clean['Entry_ID'] == entry_id_to_check]

if not entry_id_data.empty:
    print(f"Details for Entry ID {entry_id_to_check} in df_clean:")
    display(entry_id_data[['Entry_ID', 'Date', 'Saler', 'Total', 'text_for_embedding']])
else:
    print(f"Entry ID {entry_id_to_check} not found in df_clean.")

## Step 8 — AI Response Generation


### Integrate Your LLM

Below, you can add your code to integrate your preferred LLM (e.g., OpenAI, Gemini, Hugging Face models) to process the `retrieved_contexts` and generate an answer to the user's `query`. Remember, the goal is to leverage the retrieved information to provide an accurate and relevant response.

In [ ]:
# Example: Placeholder for your LLM integration
# You will need to replace this with actual code for your chosen LLM.

def generate_ai_response(query: str, retrieved_contexts: list):
    """Generates an AI response using an LLM, augmented with retrieved contexts."""
    # Concatenate the retrieved documents into a single string
    context_str = "\n\n".join([item['document'] for item in retrieved_contexts])

    # --- Your LLM Integration Goes Here ---
    # Example with a hypothetical LLM client:
    # from your_llm_library import LLMClient
    # llm_client = LLMClient(api_key="YOUR_API_KEY")

    # system_prompt = (
    #     "You are an AI assistant specialized in retail transaction analysis. "
    #     "Answer the user's query based *only* on the provided retail transaction data. "
    #     "If the answer is not in the context, state that you don't have enough information. "
    #     "Be concise and directly answer the question."
    # )

    # user_message = f"Based on the following retail transaction data:\n\n{context_str}\n\nAnswer this question: {query}"

    # response = llm_client.generate_text(
    #     prompt=user_message,
    #     system_prompt=system_prompt,
    #     max_tokens=150
    # )
    # return response.text

    # For now, return a message indicating LLM integration is pending
    return f"LLM response pending: Your LLM should process the query '{query}' with context:\n{context_str}"

# Assuming `query` and `retrieved_contexts` are still available from the previous cell's execution
final_ai_response = generate_ai_response(query, retrieved_contexts)
print("\n--- AI Generated Response ---")
print(final_ai_response)
